# Fed-BioMed to train a federated logistic regression model

## Data 


This tutorial shows how to deploy in Fed-BioMed to solve a federated logistic regression problem with scikit-learn.

In this tutorial we are using the wrapper of Fed-BioMed for the [SGD classifier](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html).
The goal of the notebook is to train a model on a realistic dataset of (synthetic) medical information mimicking the [ADNI dataset](http://adni.loni.usc.edu/).

## Creating nodes

To proceed with the tutorial, we create 2 clients with corresponding dataframes of clinical information in .csv format. Each client has data points composed by several features corresponding to clinical and medical imaging information. The data is entirely synthetic and randomly sampled to mimic the variability of the real ADNI dataset, but then **clients are purposely unbalanced, one containing all FAQ.bl values over the threshold, the other containing the FAQ.bl values below the threshold**.

The training partitions are available at the following link:

https://drive.google.com/file/d/1LhubshfUdcZVk2tbOKDsTU_Cpdo-2sjk/view?usp=sharing

The federated task we aim at solve is to predict a clinical variable (FAQ.bl, Functional Activities Questionnaire score, which  measures ability to perform daily activities) from a combination of a combination of other cognitive tests results (ADAS, MMSE, RAVLT) and MRI atrophy measures (hippocampus and ventricles volume). This is an easy task as FAQ and CDRSB measure overlapping clinical constructs.

The logistic regressor variables are the following features:

```python
['CDRSB.bl', 'ADAS11.bl', 'MMSE.bl', 'RAVLT.immediate.bl', 'RAVLT.learning.bl', 'RAVLT.forgetting.bl', 'Hippocampus.bl', 'Ventricles.bl']
```
and the target variable is:
```python
['FAQ.bl']
```    

To create the federated dataset, we follow the standard procedure for node creation/population of Fed-BioMed. 

we create a first node by using the commands

```shell
$ fedbiomed node -p my-node start
```

We then populate the node with the data of first client:

```shell
$ fedbiomed node -d my-node dataset add`
``` 
We select option 1 (csv) to add the .csv partition of client 1, by just picking the .csv of client 1. We use `adni-unbalanced` as tag to save the selected dataset. We can further check that the data has been added by executing `fedbiomed node -d my-node dataset list`

<div class="note">
    <p>Following the same procedure, we create the other two nodes with the dataset of client 2.</p>
</div>

## Fed-BioMed Researcher

We are now ready to start the researcher environment with the following command. This command will activate researcher environment and start Jupyter Notebook.
```shell
$ fedbiomed researcher start
```

## Create an experiment to train a model on the data found

The class `FedSGDClassifier` constitutes the Fed-BioMed wrapper for executing Federated Learning using Scikit-Learn `SGDClassifier` linear classifier with SGD training model. We create a new training plan class `LogisticRegressorTrainingPlan` that inherits from it. For a refresher on how Training Plans work in Fed-BioMed, please refer to our [Training Plan user guide](../../../user-guide/researcher/training-plan).

In scikit-learn Training Plans, you typically need to define only the `training_data` function, and optionally an `init_dependencies` function if your code requires additional module imports. 

The `training_data` function defines how datasets should be loaded in nodes to make them ready for training. It takes a `batch_size` argument and returns a `DataManager` class. For scikit-learn, the `DataManager` must be instantiated with a `dataset` and a `target` argument, both `np.ndarrays` of the same length.

We note that this model performs a common standardization across federated datasets by **centering with respect to the same parameters**.

In [ ]:
import numpy as np
from fedbiomed.common.training_plans import FedSGDClassifier
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.dataset import TabularDataset
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

class LogisticRegressorTrainingPlan(FedSGDClassifier):
    
    def init_dependencies(self):
        """Define additional dependencies."""
        return ["from sklearn.pipeline import Pipeline",
                "from sklearn.preprocessing import FunctionTransformer",
                "from fedbiomed.common.dataset import TabularDataset",
                "import numpy as np"]
        
    def training_data(self):
        # estimated mean and standard deviation for normalizing dataset
        scaling_mean = np.array([1.25, 9.87, 26.9, 36.6, 4.1, 4.6, 4.6e-03, 2.5e-02])
        scaling_sd = np.array([1.3, 6.0, 4.4, 12.2, 2.6, 2.3, 8e-04, 1.2e-02])

        scaler_function = lambda x: (x - scaling_mean) / scaling_sd
        
        pipeline = Pipeline([
            ('scaler', FunctionTransformer(scaler_function, validate=False))
        ])

        def apply_threshold(data):
            # we want to classify FAQ.bl value under/over threshold
            threshold = 4
            return np.array(data <= threshold, dtype=int)

        dataset = TabularDataset(input_columns=[
                                    'CDRSB.bl', 'ADAS11.bl', 'MMSE.bl',
                                    'RAVLT.immediate.bl', 'RAVLT.learning.bl', 'RAVLT.forgetting.bl',
                                    'Hippocampus.bl', 'Ventricles.bl',
                                 ], 
                                 target_columns=['FAQ.bl'],        
                                 transform=pipeline.transform,
                                 target_transform=apply_threshold,
                                )

        return DataManager(dataset=dataset)

Provide dynamic arguments for the model and training. These may potentially be changed at every round.

### Model arguments

`model_args` is a dictionary with the arguments related to the model, that will be passed to the `SGDClassifier` constructor.

**IMPORTANT** For classification tasks, you are **required** to specify the following field:
- `n_features`: the number of features in each input sample
- `n_classes`: the number of classes predicted

### Training arguments

`training_args` is a dictionary containing the arguments for the training routine (e.g. batch size, learning rate, epochs, etc.). This will be passed to the routine on the node side.

In [ ]:
from fedbiomed.common.metrics import MetricTypes

model_args = {
    # Use log_loss to do a logistic regression
    'loss': 'log_loss',
    # 2 classes to have 0/1 classification
    'n_classes': 2,
    'learning_rate': 'constant',
    'eta0':0.01,
    'n_features': 8,
}

training_args = {
    #'epochs': 1,
    'num_updates': 5,
    'loader_args': { 'batch_size': 8, },
#    'batch_maxnum': 2,  # can be used to debugging to limit the number of batches per epoch
#    'log_interval': 1,  # output a logging message every log_interval batches
}

The experiment can be now defined, by providing the `adni-unbalanced` tag, and running the local training on nodes with training plan defined in `training_plan_path`, standard `aggregator` (FedAvg) and `client_selection_strategy` (all nodes used). Federated learning is going to be performed through 20 optimization rounds.

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage

tags =  ['adni-unbalanced']
rounds = 20

# select nodes participating in this experiment
exp = Experiment(tags=tags,
                 model_args=model_args,
                 training_plan_class=LogisticRegressorTrainingPlan,
                 training_args=training_args,
                 round_limit=rounds,
                 aggregator=FedAverage(),
                 tensorboard=True,
                 node_selection_strategy=None)

In [ ]:
%load_ext tensorboard
tensorboard_dir = exp.tensorboard_results_path
%tensorboard --logdir "$tensorboard_dir"

In [ ]:
#exp.training_plan_file()

In [ ]:
# start federated training
exp.run(increase=True)

Save trained model to file

In [ ]:
exp.training_plan().export_model('./trained_model')

##  Testing

Once the federated model is obtained, it is possible to test it locally on an independent testing partition.
The test dataset is available at this link:

https://drive.google.com/uc?id=19kxuI146WA2fhcOU2_AvF8dy-ppJkzW7

In [ ]:
!pip install matplotlib
!pip install gdown

Download the testing dataset on the local temporary folder.

In [ ]:
import os
import gdown
import tempfile
import zipfile
import pandas as pd
import numpy as np

from fedbiomed.researcher.config import config

resource = "https://drive.google.com/uc?id=19kxuI146WA2fhcOU2_AvF8dy-ppJkzW7"

tmpdir = tempfile.TemporaryDirectory(dir=config.vars['TMP_DIR'])
base_dir = tmpdir.name

test_file = os.path.join(base_dir, "test_data.zip")
gdown.download(resource, test_file, quiet=False)

zf = zipfile.ZipFile(test_file)

for file in zf.infolist():
    zf.extract(file, base_dir)

# loading testing dataset
test_data = pd.read_csv(os.path.join(base_dir,'adni_validation.csv'))

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
%matplotlib inline

Here we extract the relevant input and target from the testing data 

In [ ]:
input_col=[
    'CDRSB.bl', 'ADAS11.bl', 'MMSE.bl',
    'RAVLT.immediate.bl', 'RAVLT.learning.bl', 'RAVLT.forgetting.bl',
    'Hippocampus.bl', 'Ventricles.bl',
] 
target_col=['FAQ.bl']  

X_test = test_data[input_col].values
y_test = test_data[target_col].values

We then inspect the predictions of the final federated model on the testing data, which we previously standardize similarly to the training data, and plot results as histograms and confusion matrix.

In [ ]:
fed_model = exp.training_plan().model()

scaling_mean = np.array([1.25, 9.87, 26.9, 36.6, 4.1, 4.6, 4.6e-03, 2.5e-02])
scaling_sd = np.array([1.3, 6.0, 4.4, 12.2, 2.6, 2.3, 8e-04, 1.2e-02])

threshold = 4

y_pred = fed_model.predict((X_test-scaling_mean)/scaling_sd)
y_true = np.array(y_test <= threshold, dtype=int).flatten()

TP = np.sum((y_true == 1) & (y_pred == 1))
TN = np.sum((y_true == 0) & (y_pred == 0))
FP = np.sum((y_true == 0) & (y_pred == 1))
FN = np.sum((y_true == 1) & (y_pred == 0))

counts = [TP, TN, FP, FN]
labels = ['True Pos', 'True Neg', 'False Pos', 'False Neg']

plt.bar(labels, counts, color=['green','blue','red','orange'])
plt.ylabel('Count')
plt.title('Classification Results')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Pred 0','Pred 1'], yticklabels=['True 0','True 1'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

## Compare to local training

To compare federated learning results with centrally trained model, we now train locally a model with same data as both clients. Then, we predict on test data using the local model, and plot the results.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import log_loss
from sklearn.utils.class_weight import compute_class_weight

# Change path to the location of ADNI dataset files
df = pd.concat([
    pd.read_csv('/home/mvesin/data/adni_synth_clients_unbalanced/adni_high_faq.csv'),
    pd.read_csv('/home/mvesin/data/adni_synth_clients_unbalanced/adni_low_faq.csv'),
], ignore_index=True)

input_columns = [
    'CDRSB.bl', 'ADAS11.bl', 'MMSE.bl',
    'RAVLT.immediate.bl', 'RAVLT.learning.bl', 'RAVLT.forgetting.bl',
    'Hippocampus.bl', 'Ventricles.bl',
]
target_column = 'FAQ.bl'

X_raw = df[input_columns].values
y_raw = df[target_column].values

def apply_threshold(data):
    threshold = 4
    return np.array(data <= threshold, dtype=int)

y = apply_threshold(y_raw)

# same standardization as federated
scaling_mean = np.array([1.25, 9.87, 26.9, 36.6, 4.1, 4.6, 4.6e-03, 2.5e-02])
scaling_sd = np.array([1.3, 6.0, 4.4, 12.2, 2.6, 2.3, 8e-04, 1.2e-02])
scaler_function = lambda x: (x - scaling_mean) / scaling_sd

pipeline = Pipeline([
    ('scaler', FunctionTransformer(scaler_function, validate=False))
])
X = pipeline.fit_transform(X_raw)

classes = np.array([0, 1])
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)
class_weight_dict = dict(zip(classes, weights))

local_model = SGDClassifier(
    loss="log_loss",
    class_weight=class_weight_dict,
    learning_rate="constant",
    eta0=0.01,
    max_iter=20,
)

local_model.fit(X,y)

In [ ]:
threshold = 4

y_pred_local = local_model.predict((X_test-scaling_mean)/scaling_sd)

TP = np.sum((y_true == 1) & (y_pred_local == 1))
TN = np.sum((y_true == 0) & (y_pred_local == 0))
FP = np.sum((y_true == 0) & (y_pred_local == 1))
FN = np.sum((y_true == 1) & (y_pred_local == 0))

counts = [TP, TN, FP, FN]
labels = ['True Pos', 'True Neg', 'False Pos', 'False Neg']

plt.bar(labels, counts, color=['green','blue','red','orange'])
plt.ylabel('Count')
plt.title('Classification Results')
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true, y_pred_local)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Pred 0','Pred 1'], yticklabels=['True 0','True 1'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()